In [2]:
import os
import zipfile
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
from cnnClassifier import logger
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, get_size

In [3]:
pwd

'd:\\workingfolder\\Casting_Image_Classification\\Project\\research'

In [4]:
os.chdir("../")

In [5]:
pwd

'd:\\workingfolder\\Casting_Image_Classification\\Project'

In [6]:
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
@dataclass(frozen=True)
class KaggleToken:
    username: str
    key: str    

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        secrets_filepath = TOKEN_KEY_PATH,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.secrets = read_yaml(secrets_filepath)
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

    def get_data_ingestion_secrets(self) -> KaggleToken:
        secrets = self.secrets.kaggle_secrets

        get_data_ingestion_secrets = KaggleToken(
            username=secrets.username,
            key=secrets.key
        )

        return get_data_ingestion_secrets

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig, token: KaggleToken):
        self.config = config
        self.secrets = token
    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            os.environ['KAGGLE_USERNAME'] = self.secrets.username
            os.environ['KAGGLE_KEY'] = self.secrets.key
            url = self.config.source_URL            
            kaggle.api.authenticate()
            kaggle.api.dataset_download_files(url,
                                             path=self.config.unzip_dir,
                                             unzip=False)
            
            logger.info(f" Casting dataset is downloaded!")
        else:
            logger.info(f"File already exist and has a size of: {get_size(Path(self.config.local_data_file))}")


    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            logger.info(f"Downloaded zipfile has been extracted")

In [10]:
import kaggle 

Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token


In [11]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion_secrets = config.get_data_ingestion_secrets()
    data_ingestion = DataIngestion(config=data_ingestion_config, token=data_ingestion_secrets)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-10-27 20:12:10,199: INFO: common: yaml file: token_keys\secret.yaml loaded successfully]
[2026-10-27 20:12:10,201: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-27 20:12:10,203: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-27 20:12:10,204: INFO: common: created directory at: artifacts]
[2026-10-27 20:12:10,205: INFO: common: created directory at: artifacts/data_ingestion]
Dataset URL: https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product
[2026-10-27 20:12:20,589: INFO: 635440269:  Casting dataset is downloaded!]
[2026-10-27 20:12:29,176: INFO: 635440269: Downloaded zipfile has been extracted]
